In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

DATA_PATH = "/content/drive/MyDrive/AKI Project"

os.listdir(DATA_PATH)

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [ ]:
diag_path = os.path.join(DATA_PATH, "diagnosis.csv")

diagnosis = pd.read_csv(diag_path)

diagnosis.head()

In [ ]:
diagnosis.shape
diagnosis['code_system'].value_counts()

In [ ]:
# AKI ICD-10 (N17*)
aki_icd10 = diagnosis[
    (diagnosis['code_system'] == 'ICD-10-CM') &
    (diagnosis['code'].str.startswith('N17', na=False))
]

# AKI ICD-9 (584*)
aki_icd9 = diagnosis[
    (diagnosis['code_system'] == 'ICD-9-CM') &
    (diagnosis['code'].str.startswith('584', na=False))
]

print("AKI ICD-10 count:", aki_icd10.shape[0])
print("AKI ICD-9 count:", aki_icd9.shape[0])

In [ ]:
# Combine ICD-10 and ICD-9 AKI rows
aki_all = pd.concat([aki_icd10, aki_icd9])

print("Total AKI diagnosis rows:", aki_all.shape[0])

In [ ]:
aki_all['date'] = pd.to_datetime(aki_all['date'], format='%Y%m%d', errors='coerce')

aki_all.head()

In [ ]:
aki_index = (
    aki_all
    .sort_values('date')
    .groupby('patient_id')
    .first()
    .reset_index()[['patient_id', 'date']]
)

aki_index.rename(columns={'date': 'aki_index_date'}, inplace=True)

print("Number of unique AKI patients:", aki_index.shape[0])
aki_index.head()

# EXCLUDING ESRD PATIENTS DIAGNOSED BEFORE AKI

Patients diagnosed with ESRD before or on the AKI index date are excluded because:

- ESRD represents the outcome (end-stage kidney failure).
- The study aims to predict **future** progression to ESRD after AKI.
- Including prior ESRD would violate temporal order.
- It would introduce data leakage and artificially inflate model performance.

This ensures a clean and clinically valid AKI progression cohort.

In [ ]:
# ESRD ICD-10 (exact match)
esrd_icd10 = diagnosis[
    (diagnosis['code_system'] == 'ICD-10-CM') &
    (diagnosis['code'] == 'N18.6')
]

# ESRD ICD-9 (exact match)
esrd_icd9 = diagnosis[
    (diagnosis['code_system'] == 'ICD-9-CM') &
    (diagnosis['code'] == '585.6')
]

esrd_all = pd.concat([esrd_icd10, esrd_icd9])

print("Total ESRD diagnosis rows:", esrd_all.shape[0])

In [ ]:
# Ensure AKI index date is datetime
aki_index['aki_index_date'] = pd.to_datetime(aki_index['aki_index_date'], errors='coerce')

# Ensure ESRD date is datetime
esrd_all['date'] = pd.to_datetime(esrd_all['date'], format='%Y%m%d', errors='coerce')

In [ ]:
esrd_in_aki = esrd_all.merge(aki_index, on='patient_id', how='inner')
print("ESRD rows among AKI patients:", esrd_in_aki.shape[0])
print("AKI patients with ANY ESRD at any time:", esrd_in_aki['patient_id'].nunique())

In [ ]:
pre_existing_esrd = esrd_in_aki[esrd_in_aki['date'] <= esrd_in_aki['aki_index_date']]

print("Pre-existing ESRD rows (<= index):", pre_existing_esrd.shape[0])
print("Patients with pre-existing ESRD:", pre_existing_esrd['patient_id'].nunique())

In [ ]:
exclude_patients = set(pre_existing_esrd['patient_id'].unique())

aki_clean = aki_index[~aki_index['patient_id'].isin(exclude_patients)].copy()

print("AKI patients before exclusion:", aki_index.shape[0])
print("AKI patients after ESRD exclusion:", aki_clean.shape[0])

In [ ]:
print("Any excluded patients still in cohort?",
      aki_clean['patient_id'].isin(exclude_patients).any())

# Defining 2 year prediction window

### index date and end date for each patient

In [ ]:
aki_clean['prediction_end_date'] = aki_clean['aki_index_date'] + pd.DateOffset(years=2)

aki_clean.head()

In [ ]:
aki_clean.shape[0]

In [ ]:
# Merge ESRD with clean AKI cohort
esrd_future = esrd_all.merge(aki_clean, on='patient_id', how='inner')

# Keep ESRD that happens AFTER index and WITHIN 2 years
esrd_future = esrd_future[
    (esrd_future['date'] > esrd_future['aki_index_date']) &
    (esrd_future['date'] <= esrd_future['prediction_end_date'])
]

print("Patients who progressed to ESRD within 2 years:",
      esrd_future['patient_id'].nunique())

In [ ]:
# creating a supervised learning target
# creating outcome label: 1 if progressed to ESRD within 2 years, else 0
esrd_patients_2yr = set(esrd_future['patient_id'].unique())

aki_clean['Y_esrd_2yr'] = aki_clean['patient_id'].isin(esrd_patients_2yr).astype(int)

print(aki_clean['Y_esrd_2yr'].value_counts())
print("Positive rate:", aki_clean['Y_esrd_2yr'].mean())

Loading patient.csv to inspect code systems

In [ ]:
proc_path = os.path.join(DATA_PATH, "procedure.csv")

# Use dtype=str so codes like '585.6' or codes with leading zeros don't get mangled
procedure = pd.read_csv(proc_path, dtype=str)

print(procedure.shape)
print(procedure['code_system'].value_counts())

procedure.head()

In [ ]:
# =========================
# DIALYSIS outcome within 2 years
# =========================

# 0) Ensure datetime columns exist
aki_clean = aki_clean.copy()
aki_clean["aki_index_date"] = pd.to_datetime(aki_clean["aki_index_date"], errors="coerce")
if "prediction_end_date" not in aki_clean.columns:
    aki_clean["prediction_end_date"] = aki_clean["aki_index_date"] + pd.DateOffset(years=2)

procedure = procedure.copy()
procedure["date"] = pd.to_datetime(procedure["date"], format="%Y%m%d", errors="coerce")

# 1) Dialysis code definitions (common starters; you can expand later)
CPT_DIALYSIS = {"90935", "90937", "90945", "90947"}      # dialysis procedures
HCPCS_DIALYSIS = {"G0257"}                               # unscheduled dialysis (common)
ICD10PCS_PREFIXES = ("5A1D",)                            # hemodialysis (ICD-10-PCS)
ICD9PROC_DIALYSIS = {"39.95"}                            # hemodialysis (ICD-9 procedure)

# normalize code_system to avoid case issues
procedure["code_system_norm"] = procedure["code_system"].astype(str).str.upper().str.strip()
procedure["code_norm"] = procedure["code"].astype(str).str.strip()

In [ ]:
# 2) Pull dialysis rows from procedure table
dialysis_cpt = procedure[
    (procedure["code_system_norm"].str.contains("CPT", na=False)) &
    (procedure["code_norm"].isin(CPT_DIALYSIS))
]

dialysis_hcpcs = procedure[
    (procedure["code_system_norm"].str.contains("HCPCS", na=False)) &
    (procedure["code_norm"].isin(HCPCS_DIALYSIS))
]

dialysis_icd10pcs = procedure[
    (procedure["code_system_norm"].str.contains("ICD-10", na=False)) &
    (procedure["code_system_norm"].str.contains("PCS", na=False)) &
    (procedure["code_norm"].str.startswith(ICD10PCS_PREFIXES, na=False))
]

dialysis_icd9proc = procedure[
    (procedure["code_system_norm"].str.contains("ICD-9", na=False)) &
    (procedure["code_norm"].isin(ICD9PROC_DIALYSIS))
]

dialysis_all = pd.concat(
    [dialysis_cpt, dialysis_hcpcs, dialysis_icd10pcs, dialysis_icd9proc],
    ignore_index=True
)

print("Total dialysis procedure rows (all patients):", dialysis_all.shape[0])

In [ ]:
# 3) Restrict to AKI cohort and 2-year window
dialysis_in_aki = dialysis_all.merge(
    aki_clean[["patient_id", "aki_index_date", "prediction_end_date"]],
    on="patient_id",
    how="inner"
)

dialysis_future = dialysis_in_aki[
    (dialysis_in_aki["date"] >= dialysis_in_aki["aki_index_date"]) &
    (dialysis_in_aki["date"] <= dialysis_in_aki["prediction_end_date"])
].copy()

dialysis_patients_2yr = set(dialysis_future["patient_id"].unique())
print("AKI patients with ANY dialysis within 2 years (incl acute):", len(dialysis_patients_2yr))

In [ ]:
# 4) Create dialysis label
aki_clean["Y_dialysis_2yr"] = aki_clean["patient_id"].isin(dialysis_patients_2yr).astype(int)
print(aki_clean["Y_dialysis_2yr"].value_counts())
print("Dialysis positive rate:", aki_clean["Y_dialysis_2yr"].mean())

# 5) Combine ESRD OR dialysis label (what a prior student researcher did)
# (Assumes you already created Y_esrd_2yr; recreate esrd_patients_2yr if you need)
esrd_patients_2yr = set(aki_clean.loc[aki_clean["Y_esrd_2yr"] == 1, "patient_id"])
combined_patients_2yr = esrd_patients_2yr | dialysis_patients_2yr

aki_clean["Y_esrd_or_dialysis_2yr"] = aki_clean["patient_id"].isin(combined_patients_2yr).astype(int)
print(aki_clean["Y_esrd_or_dialysis_2yr"].value_counts())
print("Combined positive rate:", aki_clean["Y_esrd_or_dialysis_2yr"].mean())

# Building the observation window

In [ ]:
aki_clean["obs_start_date"] = aki_clean["aki_index_date"] - pd.DateOffset(years=1)

In [ ]:
aki_clean.head()

In [ ]:
aki_clean.shape

In [ ]:
# filter diagnosis in the window
diagnosis["date"] = pd.to_datetime(diagnosis["date"], format="%Y%m%d", errors="coerce")

diag_obs = diagnosis.merge(
    aki_clean[["patient_id","obs_start_date","aki_index_date"]],
    on="patient_id",
    how="inner"
)

diag_obs = diag_obs[
    (diag_obs["date"] >= diag_obs["obs_start_date"]) &
    (diag_obs["date"] < diag_obs["aki_index_date"])
]

In [ ]:
diag_obs.head(20)

In [ ]:
diag_obs["code"].nunique()

# Tabular feature engineering (simpler baseline approach)
- To make sure the cohort is correct
- to check if signals/indicators exist
- gives a benchmark model

Later I will progress to sequences like a prior student researcher did so I could compare the improvement over this simple baseline

In [ ]:
# finding the 300 top most frequent diagnoses codes and arranging them in a table with the codes as columns (300 columns)

top_codes = (
    diag_obs["code"]
    .value_counts()
    .head(300)
    .index
)

In [ ]:
diag_obs_top = diag_obs[diag_obs["code"].isin(top_codes)]

In [ ]:
diag_features = (
    diag_obs_top
    .groupby(["patient_id", "code"])
    .size()
    .unstack(fill_value=0)
)

In [ ]:
diag_features.head()

In [ ]:
# merging with the AKI cohort
X = aki_clean[["patient_id"]].merge(
    diag_features,
    on="patient_id",
    how="left"
)

X = X.fillna(0)

In [ ]:
X.head()

In [ ]:
# adding the target (1 if the patient developed esrd or needed dialysis within 2 years of AKI diagnoses)
# this becomes the main modeling dataset where columns= diseases 1 year before AKI and target= esrd/dialysis within 2 years
# i will run classification on this dataset
X["target"] = aki_clean["Y_esrd_or_dialysis_2yr"]

In [ ]:
X.head()

# ML Modeling

In [ ]:
# separating features and target
# features
X_model = X.drop(columns=["patient_id", "target"])

# target
y_model = X["target"]

print(X_model.shape)
print(y_model.shape)

In [ ]:
y_model.isna().sum()

In [ ]:
# removing nan from target variable
mask = y_model.notna()

X_model = X_model[mask]
y_model = y_model[mask]

print("New shape:", X_model.shape)

In [ ]:
# train test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y_model,
    test_size=0.2,
    random_state=42,
    stratify=y_model
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
y_model.value_counts()

In [ ]:
# starting with LR as it is the standard baseline for classification
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("F1 Score:", f1_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

### above results show the dataset is heavily imbalanced and the accuracy is misleading.
- ROC AUC being 0.5 means the model is randomly guessing.
- F1 Score: 0.003 (extremely low) because recall=0.0 which means the model almost never predicts dialysis patients

# CLASS IMBALANCE FIX
- applying class weights to fix the class imbalance issue

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("F1 Score:", f1_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

- this is a big improvement as recall 0.6 which means the model is finally detecting dialysis patients (model finds 60% of real dialysis patients)
- precision = 0.1 (only 10% of predicted dialysis are correct)
- F1 = 0.17 overall balance
- ROC AUC is still bad as the model cannot separate low-risk vs high-risk patients

# NEXT improvements
- adding procedure features
- adding medication features
- adding CKD stage features
- using better models

# EXPLORING THE LAB RESULT CSV

In [ ]:
lab = pd.read_csv(DATA_PATH + "/lab_result.csv", nrows=1)

print(lab.columns)

In [ ]:
lab_sample = pd.read_csv(DATA_PATH + "/lab_result.csv", nrows=1000)

lab_sample.head()

In [ ]:
# mapping the LOINC codes to their names
term = pd.read_csv(DATA_PATH + "/standardized_terminology.csv", nrows=1000)

term.head(10)

In [ ]:
import os

path = DATA_PATH + "/standardized_terminology.csv"

with open(path) as f:
    row_count = sum(1 for line in f) - 1   # minus header

print("Total rows:", row_count)

In [ ]:
term = pd.read_csv(DATA_PATH + "/standardized_terminology.csv", nrows=None)  # original: an extract-specific row limit (withheld)
term["code_system"].value_counts().head(20)

In [ ]:
# extract kidney-related lab tests from the LOINC table
loinc_terms = term[term["code_system"] == "LOINC"]

In [ ]:
# creatinine kidney lab
loinc_terms[loinc_terms["code_description"].str.contains("creatin", case=False, na=False)].head(10)

In [ ]:
# eGFR
loinc_terms[loinc_terms["code_description"].str.contains("glomerular|egfr", case=False, na=False)].head(10)

In [ ]:
# BUN/UREA
loinc_terms[loinc_terms["code_description"].str.contains("urea|bun", case=False, na=False)].head(10)

In [ ]:
# potassium
loinc_terms[loinc_terms["code_description"].str.contains("potassium", case=False, na=False)].head(10)

# REPLICATING A PRIOR STUDENT RESEARCHER'S PIPELINE

In [ ]:
# creating a 1 year observation window for procedure table
import pandas as pd
import os

# load procedure file
proc_path = os.path.join(DATA_PATH, "procedure.csv")
procedure = pd.read_csv(proc_path, dtype=str)

# convert dates
procedure["date"] = pd.to_datetime(procedure["date"], format="%Y%m%d", errors="coerce")
aki_clean["obs_start_date"] = pd.to_datetime(aki_clean["obs_start_date"], errors="coerce")
aki_clean["aki_index_date"] = pd.to_datetime(aki_clean["aki_index_date"], errors="coerce")

# merge with cohort dates
proc_obs = procedure.merge(
    aki_clean[["patient_id", "obs_start_date", "aki_index_date"]],
    on="patient_id",
    how="inner"
)

# keep procedures in 1-year observation window
proc_obs = proc_obs[
    (proc_obs["date"] >= proc_obs["obs_start_date"]) &
    (proc_obs["date"] < proc_obs["aki_index_date"])
]

print(proc_obs.shape)
proc_obs.head()

In [ ]:
# medication.csv observation window
# load medication file
med_path = os.path.join(DATA_PATH, "medication_drug.csv")
med = pd.read_csv(med_path, dtype=str)

# convert dates
med["start_date"] = pd.to_datetime(med["start_date"], format="%Y%m%d", errors="coerce")

# merge with AKI cohort
med_obs = med.merge(
    aki_clean[["patient_id", "obs_start_date", "aki_index_date"]],
    on="patient_id",
    how="inner"
)

# keep medications in the 1-year observation window
med_obs = med_obs[
    (med_obs["start_date"] >= med_obs["obs_start_date"]) &
    (med_obs["start_date"] < med_obs["aki_index_date"])
]

print(med_obs.shape)
med_obs.head()

In [ ]:
# diagnosis observation window
diag_obs = diagnosis.merge(
    aki_clean[["patient_id", "obs_start_date", "aki_index_date"]],
    on="patient_id",
    how="inner"
)

diag_obs = diag_obs[
    (diag_obs["date"] >= diag_obs["obs_start_date"]) &
    (diag_obs["date"] < diag_obs["aki_index_date"])
]

diag_obs.head()

In [ ]:
print(diag_obs.shape)
print(proc_obs.shape)
print(med_obs.shape)

Meaning:

[figure withheld] diagnosis events

[figure withheld] procedure events

[figure withheld] medication events

All within 1 year before AKI.

So the timeline will contain [figure withheld] events total

In [ ]:
# converting each table into the same event format so they can be merged

# diagnosis events
diagnosis_events = diag_obs[["patient_id","date","code"]].copy()
diagnosis_events["event_type"] = "diagnosis"
diagnosis_events = diagnosis_events[["patient_id","date","event_type","code"]]

# procedure events
procedure_events = proc_obs[["patient_id","date","code"]].copy()
procedure_events["event_type"] = "procedure"
procedure_events = procedure_events[["patient_id","date","event_type","code"]]

# medication events
medication_events = med_obs[["patient_id","start_date","code"]].copy()
medication_events = medication_events.rename(columns={"start_date":"date"})
medication_events["event_type"] = "medication"
medication_events = medication_events[["patient_id","date","event_type","code"]]

In [ ]:
# merging all events

all_events = pd.concat(
    [diagnosis_events, procedure_events, medication_events],
    ignore_index=True
)

print(all_events.shape)
all_events.head()

In [ ]:
# sorting events chronologically per patient
all_events = all_events.sort_values(["patient_id", "date"])

all_events.head(20)

In [ ]:
# grouping the events per patient
patient_sequences = all_events.groupby("patient_id")["code"].apply(list)

print(patient_sequences.shape)
patient_sequences.head()

In [ ]:
# removing sparse patients (very short sequences)
# computing sequence length
seq_lengths = all_events.groupby("patient_id").size()

seq_lengths.head()

In [ ]:
seq_lengths.shape

In [ ]:
# keeping patients with more than 3 events
valid_patients = seq_lengths[seq_lengths >= 3].index

In [ ]:
valid_patients.shape

In [ ]:
# filtering the timeline
all_events_filtered = all_events[all_events["patient_id"].isin(valid_patients)]

print(all_events_filtered.shape)

In [ ]:
# converting the codes into sequence tokens for the transformers
# make each event unique by combining type + code
all_events_filtered["event_token"] = (
    all_events_filtered["event_type"].astype(str) + "_" +
    all_events_filtered["code"].astype(str)
)

all_events_filtered.head()

In [ ]:
# build vocabulary and numeric sequences
# unique event tokens
unique_tokens = all_events_filtered["event_token"].unique()

# token -> integer mapping
token2id = {token: i + 1 for i, token in enumerate(unique_tokens)}   # start from 1
id2token = {i: token for token, i in token2id.items()}

print("Vocabulary size:", len(token2id))

In [ ]:
# converting each patient timeline into integer sequences
patient_sequences = (
    all_events_filtered
    .groupby("patient_id")["event_token"]
    .apply(list)
)

patient_sequences_ids = patient_sequences.apply(
    lambda seq: [token2id[token] for token in seq]
)

print(patient_sequences_ids.shape)
patient_sequences_ids.head()

In [ ]:
# making sure all_events_filtered was sorted by patient id and date before grouping
all_events = all_events.sort_values(["patient_id", "date"])

In [ ]:
# attaching the label Y to each patient sequence
# create patient-level label table
labels = aki_clean[["patient_id", "Y_esrd_or_dialysis_2yr"]].drop_duplicates()

# turn sequence series into dataframe
seq_df = patient_sequences_ids.reset_index()
seq_df.columns = ["patient_id", "sequence"]

# merge sequences with labels
seq_df = seq_df.merge(labels, on="patient_id", how="inner")

print(seq_df.shape)
seq_df.head()

In [ ]:
# train test split
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    seq_df,
    test_size=0.2,
    random_state=42,
    stratify=seq_df["Y_esrd_or_dialysis_2yr"]
)

print(train_df.shape, test_df.shape)

In [ ]:
# finding the sequence lenghts before padding
train_df["seq_len"] = train_df["sequence"].apply(len)
test_df["seq_len"] = test_df["sequence"].apply(len)

print(train_df["seq_len"].describe())
print(test_df["seq_len"].describe())

In [ ]:
# padding truncating sequences
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_SEQ_LEN = 400

X_train = pad_sequences(
    train_df["sequence"],
    maxlen=MAX_SEQ_LEN,
    padding="post",
    truncating="post"
)

X_test = pad_sequences(
    test_df["sequence"],
    maxlen=MAX_SEQ_LEN,
    padding="post",
    truncating="post"
)

y_train = train_df["Y_esrd_or_dialysis_2yr"].values
y_test = test_df["Y_esrd_or_dialysis_2yr"].values

print(X_train.shape)
print(X_test.shape)

# Building Transformers

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

VOCAB_SIZE = len(token2id) + 1   # +1 because padding token = 0
MAX_SEQ_LEN = 400
EMBED_DIM = 128
NUM_HEADS = 4
FF_DIM = 128
DROPOUT = 0.1

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim, mask_zero=True)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

def build_transformer_model(maxlen, vocab_size, embed_dim=128, num_heads=4, ff_dim=128, rate=0.1):
    inputs = layers.Input(shape=(maxlen,))
    embedding_layer = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)
    x = embedding_layer(inputs)
    transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim, rate)
    x = transformer_block(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(rate)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(rate)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = Model(inputs=inputs, outputs=outputs)
    return model

model = build_transformer_model(
    maxlen=MAX_SEQ_LEN,
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    ff_dim=FF_DIM,
    rate=DROPOUT
)

model.summary()

In [ ]:
import tensorflow as tf

def binary_focal_loss(alpha=0.75, gamma=2.0):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)

        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)

        pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        alpha_t = tf.where(tf.equal(y_true, 1), alpha, 1 - alpha)

        loss_val = -alpha_t * tf.pow(1.0 - pt, gamma) * tf.math.log(pt)
        return tf.reduce_mean(loss_val)

    return loss

In [ ]:
# compiling focal loss
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=binary_focal_loss(alpha=0.75, gamma=2.0),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_auc",
    patience=3,
    mode="max",
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# evaluate test set
y_prob = model.predict(X_test).ravel()
y_pred = (y_prob >= 0.5).astype(int)

from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("F1 Score:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))